In [1]:
pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
# point this at wherever you extracted the archive on your machine
DATA_ROOT = "/home/142401017/Downloads/20news-bydate" # <- your actual path
TRAIN_DIR = os.path.join(DATA_ROOT, "20news-bydate-train")
TEST_DIR = os.path.join(DATA_ROOT, "20news-bydate-test")
print(sorted(os.listdir(TRAIN_DIR))) # should list the 20 class folders


['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [3]:
# for file in sorted(os.listdir(TRAIN_DIR)):
#     for in_file in sorted(os.listdir(os.path.join(TRAIN_DIR,file))):
#         print(in_file)


#This is reading the stopwords document ans storing in the set 
DOWNLOADS = "/home/142401017/Downloads/"
with open(os.path.join(DOWNLOADS,"en.txt"),"r") as f:
    stop_words_set = set()
    stopwords = f.readlines()
    
    for i in range(len(stopwords)):
        stopwords[i] = stopwords[i][:-1]
        stop_words_set.add(stopwords[i])
        

print(stop_words_set)



{'once', 'again', 'using', 'yourselves', 'ones', 'as', 'new', 'though', 'relatively', 'you', 'normally', 'although', 'one', 'a', 'ours', 'que', 'liked', 'upon', 'thanks', 'within', 'any', 'meanwhile', 'often', 'after', 'other', 'everyone', 'against', 'an', 'neither', 'that', 'three', 'whose', 'follows', 'willing', 'want', 'afterwards', 'formerly', 'besides', 'at', 'selves', 'none', 'com', 'nobody', 'described', 'nine', 'somewhere', 'whole', 'behind', 'happens', 'getting', 'therein', 'specified', 'never', 'maybe', 'knows', 'inasmuch', 'hence', 'awfully', 'came', 'they', 'hereby', 'seriously', 'there', 'oh', 'was', 'now', 'somebody', 'just', 'than', 'gone', 'five', 'latterly', 'clearly', 'both', 'with', 'qv', 'particular', 'whether', 'him', 'even', 'reasonably', 'during', 'nd', 'wants', 'something', 'greetings', 'ask', 'have', 'theirs', 'allow', 'need', 'let', 'goes', 'these', 'is', 'which', 'associated', 'did', 'near', 'got', 'are', 'second', 'ignored', 'over', 'said', 'ourselves', 'som

In [22]:
#This is the cleaning process for a single process
def tokenise(text):
    body = text.split("\n\n",1)       
    lower_case_and_split = body.lower().split()
    
    ans = []
    
    for word in lower_case_and_split:
        word.strip(".,!?()[]:;")
    
        if len(word) > 1 and word.isalpha() and word:
            ans.append(word)

    return ans

In [24]:
#1a
import codecs
DATA_ROOT = "/home/142401017/Downloads/20news-bydate" # <- your actual path
TRAIN_DIR = os.path.join(DATA_ROOT, "20news-bydate-train")
TEST_DIR = os.path.join(DATA_ROOT, "20news-bydate-test")


train_data_words = []
for folder in sorted(os.listdir(TRAIN_DIR)):
    FOLDER_PATH = os.path.join(TRAIN_DIR,str(folder))
    class_words = []
    
    for file in sorted(os.listdir(FOLDER_PATH)):
        CURR_FILE_PATH = os.path.join(FOLDER_PATH,file)

        F = codecs.open(CURR_FILE_PATH,"r",encoding="utf-8",errors="ignore")
        
        text = F.read()
        tokens = tokenise(text)

        for i in tokens:
            class_words.append(i)

    train_data_words.append(class_words)

In [29]:
#loop through words in train_data_words and add it to the set - vocabulary
#use a dict to store word and class as key and the corrspo prob
vocabulary = set()
for cls in train_data_words:
    for word in cls:
        vocabulary.add(word)


In [66]:
#now what we need to do get words in class and then hw many times words occurs in class
word_counts_in_each_class = []
for cls in train_data_words:
    cnt = {}
    for word in cls:
        if word not in cnt:
            cnt[word] = 0
        cnt[word] +=1

    word_counts_in_each_class.append(cnt)



In [67]:
total_words_class_wise = []
for cls in word_counts_in_each_class:
    cnt = 0
    for word in cls:
        cnt +=1
        
    total_words_class_wise.append(cnt)
        
print(total_words_class_wise)
print(len(total_words_class_wise))
print(sum(total_words_class_wise))
    
        

[8172, 7607, 5620, 6014, 5890, 7798, 6725, 7309, 7422, 6457, 8289, 9582, 7240, 10244, 10287, 9949, 9268, 11706, 9649, 8128]
20
163356


In [68]:
#we need p(class) - files in class / total docums
classes = sorted(os.listdir(TRAIN_DIR))
class_doc_count = []

for cls in classes:
    CLASS_PATH = os.path.join(TRAIN_DIR,cls)
    cnt = 0

    for file in os.listdir(CLASS_PATH):
        FILE_PATH = os.path.join(CLASS_PATH,file)

        if FILE_PATH:
            cnt+=1
            
    class_doc_count.append(cnt)

print(class_doc_count)

[480, 584, 591, 590, 578, 593, 585, 594, 598, 597, 600, 595, 591, 594, 593, 599, 546, 564, 465, 377]


In [51]:
#get p(class) class wiswe
total_docs = sum(class_doc_count)
print(total_docs)

print()
each_class_prob = []
for i in class_doc_count:
    each_class_prob.append(i/total_docs)

print(each_class_prob)
print()
print(len(each_class_prob))

11314

[0.04242531377054976, 0.05161746508750221, 0.0522361675799894, 0.05214778150963408, 0.05108714866537034, 0.05241293972070002, 0.05170585115785752, 0.05250132579105533, 0.05285487007247658, 0.052766484002121264, 0.0530316422131872, 0.05258971186141064, 0.0522361675799894, 0.05250132579105533, 0.05241293972070002, 0.052943256142831886, 0.048258794414000356, 0.04984974368039597, 0.041099522715220084, 0.033321548523952624]

20


In [69]:
# so according to the Laplac addtive formula in the question it will look something like this 
#count of word in class + alpha / total words in class + vocabulary size

def word_class_prob(word,class_idx):
    if word in word_counts_in_each_class[class_idx]:
        cnt = word_counts_in_each_class[class_idx][word]
        
    else:
        cnt = 0

    prob = (cnt+1)/(total_words_class_wise[class_idx] + len(vocabulary))
    return prob
    
print(word_class_prob("else",2))


0.0007097232079489


In [71]:
DATA_ROOT = "/home/142401017/Downloads/20news-bydate" # <- your actual path
TRAIN_DIR = os.path.join(DATA_ROOT, "20news-bydate-train")
TEST_DIR = os.path.join(DATA_ROOT, "20news-bydate-test")
classes = classes = sorted(os.listdir(TEST_DIR))
import math

#now we have to make a predict func will calu P(word/class) and then will do max of all of them and then give class
#which it belongs to

def predict_document(words_list):
    best_class = float("-inf")
    best_score = float("-inf")

    for class_idx in range(20):
        score = math.log(each_class_prob[class_idx])

        for word in words:
            prob = word_class_prob(word,class_idx)
            score += math.log(prob)

        if score > best_score:
            best_score = score
            best_class = class_idx

    return best_class

    

In [84]:
#now do the same looping thing for test data
#accuraacy = correclty predicted/total words in test
correct = 0
total = 0
predictions = []
actual_labels = []

for cls_idx in range(20):
    CLASS_PATH = os.path.join(TEST_DIR,classes[cls_idx])

    for file in os.listdir(CLASS_PATH):
        FILE_PATH = os.path.join(CLASS_PATH,file)

        F = codecs.open(CURR_FILE_PATH,"r",encoding="utf-8",errors="ignore")
        text = F.read()

        words = tokenise(text)
        pred_class = predict_document(words)

        predictions.append(pred_class)
        actual_labels.append(cls_idx)

        if pred_class == cls_idx:
            correct +=1
        total+=1



In [86]:
print(correct)
print(total)

398
7532


In [87]:
print(f"Accuracy is : {correct/total}")

Accuracy is : 0.05284121083377589


In [91]:
import matplotlib.pyplot as plt

confustion_matrix = []

for i in range(20):
    row = []
    for j in range(20):
        row.append(0)
    confustion_matrix.append(row)

In [94]:
for i in range(len(confustion_matrix)):
    actual_class = actual_labels[i]
    pred_class = predictions[i]
    confustion_matrix[actual_class][pred_class] +=1

for row in confustion_matrix:
    print(row)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 20, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0

In [103]:
train_data_words_stop = []

def tokenise_without_stopwords(text):
    body = text.split("\n\n",1)[1]       
    lower_case_and_split = body.lower().split()
    
    ans = []
    
    for word in lower_case_and_split:
        word.strip(".,!?()[]:;")
    
        if len(word) > 1 and word.isalpha() and word not in stop_words_set:
            ans.append(word)

    return ans

classes = sorted(os.listdir(TRAIN_DIR))
for folder in classes:
    FOLDER_PATH = os.path.join(TRAIN_DIR,folder)
    
    class_words = []
    for file in sorted(os.listdir(FOLDER_PATH)):
        CURR_FILE_PATH = os.path.join(FOLDER_PATH,file)
        F = codecs.open(CURR_FILE_PATH,"r",encoding="utf-8",errors="ignore")
        text = F.read()

        words_without_tokens = tokenise_without_stopwords(text)
        class_words.append(words_without_tokens)
    train_data_words_stop.append(class_words)

    

In [113]:
vocabulary = set()
for class_docs in train_data_words_stop:
    for doc in class_docs:
        for word in doc:
            vocabulary.add(word)


In [114]:
word_counts_in_each_class = []
for class_doc in train_data_words_stop:
    class_word_count = {}
    for doc in class_doc:
        for word in doc:
            if word not in class_word_count:
                class_word_count[word] = 0

            class_word_count[word] +=1
    word_counts_in_each_class.append(class_word_count)

In [116]:
#gett total words in each class
total_words_class_wise = []
for class_docs in train_data_words_stop:
    total = 0
    for doc in class_docs:
        total += len(doc)
    total_words_class_wise.append(total)

print(total_words_class_wise)

[38739, 34863, 24925, 29523, 26190, 46676, 23263, 32349, 29251, 31838, 48699, 63552, 31689, 48824, 53264, 54305, 52188, 77897, 54459, 32829]


In [117]:
total_docs = sum(class_doc_count)
print(total_docs)

print()
each_class_prob = []
for i in class_doc_count:
    each_class_prob.append(i/total_docs)

print(each_class_prob)
print()
print(len(each_class_prob))

11314

[0.04242531377054976, 0.05161746508750221, 0.0522361675799894, 0.05214778150963408, 0.05108714866537034, 0.05241293972070002, 0.05170585115785752, 0.05250132579105533, 0.05285487007247658, 0.052766484002121264, 0.0530316422131872, 0.05258971186141064, 0.0522361675799894, 0.05250132579105533, 0.05241293972070002, 0.052943256142831886, 0.048258794414000356, 0.04984974368039597, 0.041099522715220084, 0.033321548523952624]

20


In [118]:
def word_class_prob(word,class_idx):
    if word in word_counts_in_each_class[class_idx]:
        cnt = word_counts_in_each_class[class_idx][word]
        
    else:
        cnt = 0

    prob = (cnt+1)/(total_words_class_wise[class_idx] + len(vocabulary))
    return prob
    
print(word_class_prob("else",2))


1.305721672368318e-05


In [120]:
def predict_document(words_list):
    best_class = float("-inf")
    best_score = float("-inf")

    for class_idx in range(20):
        score = math.log(each_class_prob[class_idx])

        for word in words:
            prob = word_class_prob(word,class_idx)
            score += math.log(prob)

        if score > best_score:
            best_score = score
            best_class = class_idx

    return best_class

In [124]:
correct = 0
total = 0


for cls_idx in range(20):
    CLASS_PATH = os.path.join(TEST_DIR,classes[cls_idx])
    
    for file in sorted(os.listdir(CLASS_PATH)):
        CURR_FILE_PATH = os.path.join(CLASS_PATH,file)
        
        F = codecs.open(CURR_FILE_PATH,"r",encoding="utf-8",errors="ignore")
        text = F.read()

        words_without_tokens = tokenise_without_stopwords(text)
        pred_class = predict_document(words_without_tokens)

        if pred_class == cls_idx:
            correct +=1

        total +=1
        

            
        

In [125]:
print(f"Accuracy is {correct/total}")

Accuracy is 0.03332448220924057


In [126]:
confusion_matrix = []

for i in range(20):
    row = []
    for j in range(20):
        row.append(0)
    confusion_matrix.append(row)

for i in range(len(predictions)):
    actual_class = actual_labels[i]
    pred_class = predictions[i]
    confustion_matrix[actual_class][pred_class] +=1

for row in confustion_matrix:
    print(row)


[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 339, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 389, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 394, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 392, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 385, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 395, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 390, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 396, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 398, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 397, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 399, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 396, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 393, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 396, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 394, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 398, 0, 0

In [158]:
word_counts_in_each_class = []

for class_doc in train_data_words_stop:
    class_word_count = {}
    for doc in class_doc:
        for word in doc:
            if word not in class_word_count:
                class_word_count[word] = 0

            class_word_count[word] +=1
    word_counts_in_each_class.append(class_word_count)
    
def word_class_prob(word,class_idx,aplha):
    if word in word_counts_in_each_class[class_idx]:
        cnt = word_counts_in_each_class[class_idx][word]
    else:
        cnt = 0
    prob = (cnt+aplha)/(total_words_class_wise[class_idx]+aplha*len(vocabulary))
    return prob

In [159]:
def predict_document(words_list):
    best_class = float("-inf")
    best_score = float("-inf")

    for class_idx in range(20):
        score = math.log(each_class_prob[class_idx])

        for word in words:
            prob = word_class_prob(word,class_idx,alpha)
            score += math.log(prob)

        if score > best_score:
            best_score = score
            best_class = class_idx

    return best_class

In [163]:
alphas = [0.0001,0.01,0.1,1,10]
original_res = []
for alpha in alphas:
    correct = 0
    total = 0

    for cls_idx in range(20):
        CLASS_PATH = os.path.join(TEST_DIR,classes[cls_idx])

        for file in sorted(os.listdir(CLASS_PATH)):
            CURR_FILE_PATH = os.path.join(CLASS_PATH,file)

            F = codecs.open(CURR_FILE_PATH,"r",encoding="utf-8",errors="ignore")
            text = F.read()
    
            words_without_tokens = tokenise_without_stopwords(text)
            pred_class = predict_document(words_without_tokens)
    
            if pred_class == cls_idx:
                correct +=1
    
            total +=1

    accuracy = correct/total
    original_res.append(accuracy)

In [164]:
print(original_res)

[0.03332448220924057, 0.03332448220924057, 0.03332448220924057, 0.03332448220924057, 0.05284121083377589]
